# Logging in Data Science Projects

## Table of Contents

1. [Why print() is Not Enough](#1)
2. [Python logging Module](#2)
3. [Log Levels and When to Use Them](#3)
4. [Structured JSON Logging](#4)
5. [loguru: A Cleaner Alternative](#5)
6. [Exercise: Log a Training Run](#6)

---
## 1. Why print() is Not Enough <a id="1"></a>

When a long training run fails at 3am, you need to know:
- *What was the data state when it failed?*
- *Which step of the pipeline was it on?*
- *Was there a warning earlier that predicted this?*

`print()` gives you none of that. It has no timestamps, no severity levels, no way to write to a file, no way to silence it in production without deleting lines.

**Logging gives you all of that.**

In [ ]:
import logging
import sys

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    stream=sys.stdout,
)

# Best practice: always use a named logger — never the root logger
logger = logging.getLogger("churn_pipeline")

logger.debug("Detailed diagnostic info")
logger.info("Pipeline step completed: 10,432 rows loaded")
logger.warning("3 rows dropped: missing TotalCharges")
logger.error("Training failed: too few samples in minority class")

---
## 2. Python logging Module <a id="2"></a>

The architecture: **loggers** emit records, **handlers** route them (console, file, HTTP), **formatters** control the text format.

In [ ]:
import logging
from pathlib import Path


def setup_logging(name: str, log_file: str = None, level: int = logging.INFO) -> logging.Logger:
    """Create a named logger with console and optional file output.

    Args:
        name: Logger name — use __name__ of the calling module.
        log_file: Optional path for file output.
        level: Minimum severity level to capture.

    Returns:
        Configured Logger instance.
    """
    logger = logging.getLogger(name)
    logger.setLevel(level)
    logger.handlers.clear()  # prevent duplicate handlers if re-run

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )

    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(formatter)
    logger.addHandler(console)

    if log_file:
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file)
        fh.setFormatter(formatter)
        logger.addHandler(fh)

    return logger


log = setup_logging("training_demo", log_file="/tmp/training.log")
log.info("Pipeline started")
log.warning("2.1%% missing values in 'tenure' — filling with median")
log.info("Training complete — accuracy: 0.891")

print()
print("File contents:")
print(open("/tmp/training.log").read())

---
## 3. Log Levels and When to Use Them <a id="3"></a>

| Level | When to use |
|---|---|
| `DEBUG` | Step-by-step diagnostics. Off in production. |
| `INFO` | Pipeline milestones. Stage transitions. Row counts. |
| `WARNING` | Data quality issues, deprecated paths, unusual values. |
| `ERROR` | A step failed. Results may be wrong or incomplete. |
| `CRITICAL` | Process cannot continue. System-level failure. |

In [ ]:
import numpy as np
import pandas as pd
import time


class DataPipeline:
    """An example pipeline that uses logging throughout."""

    def __init__(self, name: str):
        self.name = name
        self.logger = setup_logging(f"pipeline.{name}")

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.info(f"Pipeline '{self.name}' started | input: {df.shape}")
        t0 = time.time()

        df = self._validate(df)
        df = self._clean(df)
        df = self._transform(df)

        self.logger.info(f"Done in {time.time()-t0:.2f}s | output: {df.shape}")
        return df

    def _validate(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.debug("Validating required columns")
        required = ["tenure", "MonthlyCharges"]
        missing = [c for c in required if c not in df.columns]
        if missing:
            self.logger.error(f"Missing required columns: {missing}")
            raise ValueError(f"Missing: {missing}")
        return df

    def _clean(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.debug("Dropping rows with missing required values")
        n_before = len(df)
        df = df.dropna(subset=["tenure", "MonthlyCharges"])
        dropped = n_before - len(df)
        if dropped:
            self.logger.warning(f"Dropped {dropped} rows with missing required values")
        return df

    def _transform(self, df: pd.DataFrame) -> pd.DataFrame:
        self.logger.debug("Adding log-tenure feature")
        df = df.copy()
        df["tenure_log"] = np.log1p(df["tenure"])
        return df


np.random.seed(42)
df_raw = pd.DataFrame({
    "tenure":         np.append(np.random.randint(1, 60, 95), [None]*5),
    "MonthlyCharges": np.random.uniform(20, 100, 100),
})

pipeline = DataPipeline("churn_prep")
df_out = pipeline.run(df_raw)

---
## 4. Structured JSON Logging <a id="4"></a>

In production, logs are consumed by monitoring systems (Elasticsearch, Datadog, CloudWatch). Plain text is hard to query. JSON logs are machine-readable.

In [ ]:
import json
import logging
from datetime import datetime


class JSONFormatter(logging.Formatter):
    """Emit one JSON object per log line."""

    def format(self, record: logging.LogRecord) -> str:
        data = {
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "level":     record.levelname,
            "logger":    record.name,
            "message":   record.getMessage(),
        }
        if hasattr(record, "ctx"):  # allow extra context dict
            data.update(record.ctx)
        return json.dumps(data)


def json_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(f"json.{name}")
    logger.setLevel(logging.DEBUG)
    logger.handlers.clear()
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(JSONFormatter())
    logger.addHandler(h)
    return logger


jlog = json_logger("training")
jlog.info("Training started")

# Attach structured context by setting a custom attribute before logging
class ContextAdapter(logging.LoggerAdapter):
    def process(self, msg, kwargs):
        kwargs.setdefault("extra", {})
        return msg, kwargs

    def info_ctx(self, msg, ctx: dict):
        record = self.logger.makeRecord(
            self.logger.name, logging.INFO, "", 0, msg, [], None
        )
        record.ctx = ctx
        self.logger.handle(record)

jadapter = ContextAdapter(jlog)
jadapter.info_ctx("Epoch completed", {"epoch": 10, "train_loss": 0.043, "val_loss": 0.051})

---
## 5. loguru: A Cleaner Alternative <a id="5"></a>

`loguru` replaces the stdlib logging module with a much simpler API. Install with `pip install loguru`.

In [ ]:
try:
    from loguru import logger as loguru_logger
    import sys

    loguru_logger.remove()
    loguru_logger.add(
        sys.stdout,
        format="{time:YYYY-MM-DD HH:mm:ss} | {level: <8} | {name} | {message}",
        level="DEBUG",
    )
    loguru_logger.add("/tmp/ml_run.log", rotation="10 MB", retention="7 days")

    loguru_logger.info("Pipeline started")
    loguru_logger.warning("Missing values detected: {n} rows", n=12)
    loguru_logger.success("Model trained successfully")

    print()
    print("loguru advantages:")
    print("- No handler/formatter/getLogger boilerplate")
    print("- File rotation built-in")
    print("- Colourised terminal output")

except ImportError:
    print("Install loguru: pip install loguru")
    print("  from loguru import logger")
    print("  logger.info('Pipeline started')")


---
## 6. Exercise: Log a Training Run <a id="6"></a>

Refactor the `train_model_before()` function to use proper logging:

1. Create a named logger `"model_training"`
2. `INFO`: pipeline start, data shape, model params, final accuracy and elapsed time
3. `WARNING`: if class imbalance ratio > 4:1 (e.g. 80%+ majority class)
4. `ERROR`: if `n_estimators < 10` (too few trees — likely misconfiguration)

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# BEFORE: print-based
def train_model_before(X: pd.DataFrame, y: pd.Series, n_estimators: int = 100) -> float:
    print("Starting training...")
    print(f"Data: {X.shape}")
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    clf = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    print(f"Done. Accuracy: {acc:.3f}")
    return acc


# YOUR TASK: logging-based version
def train_model_logged(X: pd.DataFrame, y: pd.Series, n_estimators: int = 100) -> float:
    # Your implementation here
    pass


# Test
np.random.seed(42)
X_d = pd.DataFrame({"tenure": np.random.randint(1, 60, 300),
                    "charges": np.random.uniform(20, 100, 300)})
y_d = pd.Series((np.random.rand(300) > 0.8).astype(int))  # ~80% class 0

train_model_logged(X_d, y_d, n_estimators=50)
print()
train_model_logged(X_d, y_d, n_estimators=5)  # should log a warning or error